# Suite PROP — Randomized properties (Hypothesis)

Invariants that must hold for all inputs, checked on generated natives and random graphs. Hypothesis shrinks any
failure to a minimal example. Seeds are derandomized so runs are reproducible.

In [ ]:
import math

from hypothesis import HealthCheck, given, settings, strategies as st

from schemas.Framework import JSON, Plain, Proxies, Reachable, Schemas as S, Validators, YAML
from support import raises, same_graph, text

CHECKS = settings(max_examples=200, deadline=None, derandomize=True,
                  suppress_health_check=[HealthCheck.too_slow, HealthCheck.data_too_large])

Node = S.OfObject.Builder().properties(text("id"), text("s"), text("i", int), text("f", float), text("b", bool),
                                       text("raw", bytes)).create()
Edge = S.OfRelation.Builder().links("src", "dst").properties(text("label"), text("w", float)).create()
Node = S.OfObject.Builder(Node).relations(lambda r: r.name("out").of(Edge).me("src"),
                                          lambda r: r.name("in").of(Edge).me("dst")).update()
Proxies.register("Node", Node)
Proxies.register("Edge", Edge)
B = Proxies.Builders
validate = Validators.Validate(B)

any_text = st.text(st.characters(codec=None, exclude_categories=()), max_size=40)  # includes lone surrogates
any_float = st.floats(allow_nan=True, allow_infinity=True, allow_subnormal=True)
any_int = st.integers(min_value=-(2**200), max_value=2**200)
natives = {str: any_text, int: any_int, float: any_float, bool: st.booleans(), bytes: st.binary(max_size=64)}


def same_value(a, b):
    if isinstance(a, float) and isinstance(b, float):
        return (math.isnan(a) and math.isnan(b)) or (a == b and math.copysign(1, a) == math.copysign(1, b))
    return type(a) is type(b) and a == b

## PROP-01 · Every native value round-trips through Plain, JSON and YAML, keeping its exact type

In [ ]:
@CHECKS
@given(st.sampled_from(list(natives)).flatmap(lambda t: st.tuples(st.just(t), natives[t])))
def natives_round_trip(pair):
    native, value = pair
    schema = S.OfNative.Data(native)
    assert same_value(Plain.FromPlain(B)(schema, Plain.ToPlain(schema, value)), value)
    assert same_value(JSON.FromJSON(B)(schema, JSON.ToJSON(schema, value)), value)
    assert same_value(YAML.FromYAML(B)(schema, YAML.ToYAML(schema, value)), value)


natives_round_trip()

## PROP-02 · Lone surrogates survive both text encodings (YAML escapes them)

In [ ]:
for value in ["\ud800", "a\udfffb", "\udc00\ud800"]:
    schema = S.OfNative.Data(str)
    assert YAML.FromYAML(B)(schema, YAML.ToYAML(schema, value)) == value
    assert JSON.FromJSON(B)(schema, JSON.ToJSON(schema, value)) == value
assert YAML.ToYAML(S.OfNative.Data(str), "\ud800x") == '"\\uD800x"\n'

## PROP-03 · Mismatched native types are always rejected by `from_plain`

In [ ]:
@CHECKS
@given(st.sampled_from(list(natives)), st.sampled_from(list(natives)).flatmap(lambda t: natives[t]))
def mismatches_rejected(native, value):
    schema = S.OfNative.Data(native)
    plain = S.OfNative.Data(type(value)).to_plain(value)
    if type(plain) is native or (native is bytes and isinstance(plain, str)) or (native is float and isinstance(plain, str)):
        return  # a matching plain form (or text that may legitimately be bytes / a non-finite float)
    with raises(TypeError, ValueError):
        schema.from_plain(plain)


mismatches_rejected()

## PROP-04 · Random graphs: round trips, validity, reachability and single-object snapshots

In [ ]:
@st.composite
def graphs(draw):
    size = draw(st.integers(1, 6))
    nodes = []
    for index in range(size):
        builder = B.Node().id(str(index))
        for name in ["s", "i", "f", "b", "raw"]:
            if draw(st.booleans()):
                getattr(builder, name)(draw(natives[{"s": str, "i": int, "f": float, "b": bool, "raw": bytes}[name]]))
        nodes.append(builder.create())
    edges = draw(st.lists(st.tuples(st.integers(0, size - 1), st.integers(0, size - 1),
                                    st.none() | any_text, st.none() | any_float), max_size=10))
    for src, dst, label, weight in edges:
        def spec(x, dst=dst, label=label, weight=weight):
            x.dst(nodes[dst])
            if label is not None:
                x.label(label)
            if weight is not None:
                x.w(weight)
            return x

        B.Node(nodes[src]).out(spec).update()
    return nodes, edges


def component(nodes, edges, start):
    neighbours = {i: set() for i in range(len(nodes))}
    for src, dst, _, _ in edges:
        neighbours[src].add(dst)
        neighbours[dst].add(src)
    seen, todo = {start}, [start]
    while todo:
        for n in neighbours[todo.pop()]:
            if n not in seen:
                seen.add(n)
                todo.append(n)
    return {nodes[i].identity() for i in seen}


@CHECKS
@given(graphs())
def graph_invariants(graph):
    nodes, edges = graph
    root = nodes[0]
    snapshot = Plain.ToPlain.Reachable(Node, root)

    assert validate.Reachable(Node, root) == []
    reached = Reachable.of(root)
    assert reached[0] is root and {o.identity() for o in reached} == component(nodes, edges, 0)
    assert len(snapshot["objects"]) == len(reached)
    assert list(Plain.ToPlain(Node, root)["objects"]) == ["s0"]

    restored = Plain.FromPlain(B).Reachable(Node, snapshot)
    assert same_graph(Plain.ToPlain.Reachable(Node, restored), snapshot)
    assert JSON.loads(JSON.ToJSON.Reachable(Node, root)) == snapshot
    assert same_graph(Plain.ToPlain.Reachable(Node, JSON.FromJSON(B).Reachable(Node, JSON.ToJSON.Reachable(Node, root))),
                      snapshot)
    text_ = YAML.ToYAML.Reachable(Node, root)
    assert same_graph(YAML.loads(text_), snapshot)
    assert same_graph(Plain.ToPlain.Reachable(Node, YAML.FromYAML(B).Reachable(Node, text_)), snapshot)


graph_invariants()

## PROP-05 · Entries form a set: the count equals the number of distinct entries under schema equality

In [ ]:
def key(value):
    if isinstance(value, float):
        return ("float", value.hex())
    return (type(value).__name__, value)


@CHECKS
@given(st.lists(st.tuples(st.none() | st.sampled_from(["a", "b"]),
                          st.none() | st.sampled_from([0.0, -0.0, math.nan, 1.0])), max_size=12))
def entry_set(props):
    owner, target = B.Node().id("o").create(), B.Node().id("t").create()
    for label, weight in props:
        def spec(x, label=label, weight=weight):
            x.dst(target)
            if label is not None:
                x.label(label)
            if weight is not None:
                x.w(weight)
            return x

        B.Node(owner).out(spec).update()
    distinct = {(key(label) if label is not None else None, key(w) if w is not None else None) for label, w in props}
    written = Plain.ToPlain(Node, owner)["objects"]["s0"].get("out", [])
    assert len(written) == len(distinct)


entry_set()